# 05 ? Multilingual NLP preprocessing

Inspect the actual Phase 1 CSVs, validate required fields, normalize conservatively, and flag sensitive patterns without printing matched values. Original processed files remain unchanged.

In [ ]:
from pathlib import Path
import pandas as pd
from src.nlp.preprocessing import normalize_text, detect_language

ROOT = Path("data/processed")
queries = pd.read_csv(ROOT / "citizen_queries.csv", keep_default_na=False)
complaints = pd.read_csv(ROOT / "complaint_queries.csv", keep_default_na=False)
required = {"query", "language", "intent", "source_type"}
assert required <= set(queries.columns), f"Missing: {required - set(queries.columns)}"
assert {"query", "language", "complaint_category"} <= set(complaints.columns)
print("citizen queries", queries.shape, "complaints", complaints.shape)
print("columns", list(queries.columns))
print("missing citizen values", queries.isna().sum().to_dict())
print("exact duplicate query rows", int(queries["query"].duplicated().sum()))
print("language distribution\n", queries["language"].value_counts())
print("intent distribution\n", queries["intent"].value_counts())
print("complaint categories\n", complaints["complaint_category"].value_counts())
print("complaint columns", list(complaints.columns))
print("missing complaint values", complaints.isna().sum().to_dict())
print("exact duplicate complaint rows", int(complaints["query"].duplicated().sum()))

In [ ]:
queries["normalized_query"] = queries["query"].map(normalize_text)
queries["script_language_guess"] = queries["normalized_query"].map(detect_language)
# Annotated language labels are primary; this comparison is a diagnostic only.
print(pd.crosstab(queries["language"], queries["script_language_guess"], margins=True))
print("empty queries", int((queries["normalized_query"] == "").sum()))
# Return only counts: never display potentially sensitive matched values.
pii_pattern = r"(?i)(\b\d{10,12}\b|\b[A-Z]{5}\d{4}[A-Z]\b|\b[^\s@]+@[^\s@]+\.[^\s@]+)"
pii_suspect = queries["normalized_query"].str.contains(pii_pattern, regex=True)
print("PII-like rows flagged for review:", int(pii_suspect.sum()))
# Do not save a new training copy until flagged records receive privacy review.
complaints["normalized_query"] = complaints["query"].map(normalize_text)
complaint_pii_suspect = complaints["normalized_query"].str.contains(pii_pattern, regex=True)
print("PII-like complaint rows flagged for review:", int(complaint_pii_suspect.sum()))
# Display examples only after screening; the scan is heuristic and needs human review if nonzero.
if not pii_suspect.any():
    for language, group in queries.groupby("language"):
        print(language, group["normalized_query"].iloc[0])